In [0]:
# =========================================================
# OPENBREWERY - CONFIG NOTEBOOK
# =========================================================
# OBJETIVO:
# Criar toda infraestrutura do projeto:
#
# - Catálogo
# - Schemas
# - External Volumes
# - External Tables
# - Control Table
# - Variáveis reutilizáveis
#
# =========================================================

In [0]:
# =========================================================
# CONFIGURAÇÕES GERAIS
# =========================================================

catalog_name = "brewing"

storage_account = "sabeesdevwestus001"

landing_container = "bees-landing-dev-001"
bronze_container = "bees-bronze-dev-001"
silver_container = "bees-silver-dev-001"
gold_container = "bees-gold-dev-001"
config_container = "bees-config-dev-001"

# =========================================================
# PATHS - VOLUMES
# =========================================================

landing_volume_path = (
    f"abfss://{landing_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/landing/"
)

config_volume_path = (
    f"abfss://{config_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/config/"
)

# =========================================================
# PATHS - TABELAS
# =========================================================

bronze_table_path = (
    f"abfss://{bronze_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/bronze/tables/openbrewery_bronze"
)

silver_table_path = (
    f"abfss://{silver_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/silver/tables/openbrewery_silver"
)

# =========================================================
# VARIÁVEIS REUTILIZÁVEIS
# =========================================================

landing_volume = (
    f"/Volumes/{catalog_name}/landing/"
    "openbrewery_landing_volume"
)

config_volume = (
    f"/Volumes/{catalog_name}/config/"
    "openbrewery_config_volume"
)

schema_location = (
    f"{config_volume}/schemas/openbrewery"
)

checkpoint_path = (
    f"{config_volume}/checkpoints/openbrewery"
)

bronze_table = (
    f"{catalog_name}.bronze.openbrewery_bronze"
)

silver_table = (
    f"{catalog_name}.silver.openbrewery_silver"
)

control_table = (
    f"{catalog_name}.control.openbrewery_control"
)

print("Variáveis carregadas com sucesso.")

In [0]:
%sql
-- =========================================================
-- REMOVE CATÁLOGO ANTIGO
-- =========================================================

DROP CATALOG IF EXISTS brewing CASCADE;

In [0]:
%sql
-- =========================================================
-- CRIA CATÁLOGO
-- =========================================================

CREATE CATALOG IF NOT EXISTS brewing;

In [0]:
%sql
-- =========================================================
-- CRIA SCHEMAS
-- =========================================================

CREATE SCHEMA IF NOT EXISTS brewing.landing;
CREATE SCHEMA IF NOT EXISTS brewing.bronze;
CREATE SCHEMA IF NOT EXISTS brewing.silver;
CREATE SCHEMA IF NOT EXISTS brewing.gold;
CREATE SCHEMA IF NOT EXISTS brewing.config;
CREATE SCHEMA IF NOT EXISTS brewing.control;

In [0]:
# =========================================================
# CRIA EXTERNAL VOLUMES
# =========================================================

spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS
{catalog_name}.landing.openbrewery_landing_volume
LOCATION '{landing_volume_path}'
""")

spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS
{catalog_name}.config.openbrewery_config_volume
LOCATION '{config_volume_path}'
""")

print("Volumes criados com sucesso.")

In [0]:
# =========================================================
# CRIA TABELA BRONZE
# =========================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{bronze_table}
(

    id STRING,
    name STRING,
    brewery_type STRING,

    address_1 STRING,
    address_2 STRING,
    address_3 STRING,

    city STRING,
    state STRING,
    state_province STRING,
    postal_code STRING,
    country STRING,

    longitude DOUBLE,
    latitude DOUBLE,

    phone STRING,
    website_url STRING,
    street STRING,

    metadata STRUCT<
        ingestion_timestamp_utc: STRING,
        page: BIGINT,
        records: BIGINT,
        source: STRING
    >,

    _rescued_data STRING,

    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE,

    source_file STRING,
    source_path STRING,

    file_modification_time TIMESTAMP,
    file_size BIGINT,

    execution_id STRING,

    record_hash STRING

)
USING DELTA
PARTITIONED BY (ingestion_date)
LOCATION '{bronze_table_path}'
""")

print("Tabela Bronze criada com sucesso.")

In [0]:
# =========================================================
# CRIA TABELA SILVER
# =========================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{silver_table}
(

    id STRING,
    name STRING,
    brewery_type STRING,

    address_1 STRING,
    address_2 STRING,
    address_3 STRING,

    city STRING,
    state STRING,
    state_province STRING,
    postal_code STRING,
    country STRING,

    longitude DOUBLE,
    latitude DOUBLE,

    phone STRING,
    website_url STRING,
    street STRING,

    record_hash STRING,

    valid_from TIMESTAMP,
    valid_to TIMESTAMP,

    is_current BOOLEAN,

    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE

)
USING DELTA
LOCATION '{silver_table_path}'
""")

print("Tabela Silver criada com sucesso.")

In [0]:
# =========================================================
# CRIA TABELA DE CONTROLE
# =========================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{control_table}
(

    pipeline_name STRING,
    last_ingestion_timestamp TIMESTAMP,
    processed_at TIMESTAMP,
    records_processed BIGINT

)
USING DELTA
""")

print("Tabela Control criada com sucesso.")

In [0]:
# =========================================================
# RESUMO FINAL
# =========================================================

print(f"""

Infraestrutura criada com sucesso.

CATALOG:
- {catalog_name}

SCHEMAS:
- landing
- bronze
- silver
- gold
- config
- control

VOLUMES:
- openbrewery_landing_volume
- openbrewery_config_volume

TABELAS:
- {bronze_table}
- {silver_table}
- {control_table}

PATHS:
- Landing: {landing_volume_path}
- Bronze: {bronze_table_path}
- Silver: {silver_table_path}
- Config: {config_volume_path}

""")